# Transfer Learning with ResNet50 on TF Flowers Dataset
This notebook demonstrates transfer learning using ResNet50 to classify the TensorFlow Flowers dataset.

In [14]:
# Imports
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os

In [15]:
# Check TF and GPU
print("TF version:", tf.__version__)
print("All physical devices:")
print(tf.config.list_physical_devices())

print("\nGPU devices:")
print(tf.config.list_physical_devices("GPU"))

print("Built with GPU support:", tf.test.is_built_with_gpu_support())
print("GPU device name:", tf.test.gpu_device_name())

In [16]:
# Download flower data
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
tf.keras.utils.get_file(origin=dataset_url, fname="flower_photos", untar=True)

data_dir = os.path.expanduser("~/.keras/datasets/flower_photos/flower_photos")

print("Data folder:", data_dir)


print(os.listdir(data_dir))

In [17]:
# Build train/val sets
img_size = (224, 224)
batch_size = 128

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int'
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int'
)

In [18]:
# Show class names
class_names = train_ds.class_names
print("Class names:", class_names)

In [19]:
# Preprocess and augment
from tensorflow.keras.applications.resnet50 import preprocess_input

def preprocess(image, label):
    image = preprocess_input(image)
    return image, label

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomZoom(0.2),
])  # prevent overfitting

train_ds = train_ds.map(preprocess)
val_ds   = val_ds.map(preprocess)

In [20]:
# Prefetch batches
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [21]:
# Build the model
def build_model(train_stage="linear_probe"):
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    base_model.trainable = True
    for layer in base_model.layers:
        layer.trainable = False

    if train_stage == "stage5":
        for layer in base_model.layers:
            if layer.name.startswith("conv5_"):
                layer.trainable = True
    elif train_stage == "stage4_stage5":
        for layer in base_model.layers:
            if layer.name.startswith("conv4_") or layer.name.startswith("conv5_"):
                layer.trainable = True

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = preprocess_input(x)
    x = base_model(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    outputs = tf.keras.layers.Dense(5, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=3e-6,
            weight_decay=1e-5
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [22]:
# Train all setups
EPOCHS = 15

# 1. Linear probe
print(f"Running: Linear probe + {EPOCHS} epochs")
model_lp = build_model(train_stage="linear_probe")
hist_lp = model_lp.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

# 2. Train stage 5
print(f"Running: Stage 5 + {EPOCHS} epochs")
model_s5 = build_model(train_stage="stage5")
hist_s5 = model_s5.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

# 3. Train stages 4 and 5
print(f"Running: Stage 4 and 5 + {EPOCHS} epochs")
model_s45 = build_model(train_stage="stage4_stage5")
hist_s45 = model_s45.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

In [23]:
# Collect histories
histories = {
    "Linear Probe": hist_lp,
    "Stage 5": hist_s5,
    "Stage 4 + 5": hist_s45
}

In [24]:
# Plot results
def plot_training_curves(histories):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for name, hist in histories.items():
        epochs = range(1, len(hist.history["accuracy"]) + 1)
        axes[0].plot(epochs, hist.history["accuracy"], label=f"{name} Train")
        axes[0].plot(epochs, hist.history["val_accuracy"], '--', label=f"{name} Val")

        axes[1].plot(epochs, hist.history["loss"], label=f"{name} Train")
        axes[1].plot(epochs, hist.history["val_loss"], '--', label=f"{name} Val")

    axes[0].set_title("Accuracy vs Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].grid(True)
    axes[0].legend()

    axes[1].set_title("Loss vs Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].grid(True)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


print("Plotting accuracy and loss curves...")
plot_training_curves(histories)

In [25]:
# Show sample predictions
def show_predictions(model, dataset, class_names, num_images=9):
    plt.figure(figsize=(12, 12))

    # take a batch
    for images, labels in dataset.take(1):
        preds = model.predict(images)
        preds = np.argmax(preds, axis=1)

        for i in range(num_images):
            plt.subplot(3, 3, i + 1)
            img = images[i].numpy()

            # 從 [0,1] → 顯示
            plt.imshow(img)

            pred_name = class_names[preds[i]]
            true_name = class_names[labels[i]]

            color = "green" if pred_name == true_name else "red"
            plt.title(f"Pred: {pred_name}\nTrue: {true_name}", color=color)
            plt.axis("off")
        break

    plt.tight_layout()
    plt.show()

models = {
    "Linear Probe": model_lp,
    "Stage 5": model_s5,
    "Stage 4 + 5": model_s45
}
for name, m in models.items():
    print(f"=== {name} ===")
    show_predictions(m, val_ds, class_names)